# 11 - Bilingual Round-Trip

Translate each COT to Chinese, then back to English. This is the most aggressive
semantic bottleneck. Prefill PRIMARY_MODEL with the round-tripped COT.

If accuracy holds, the reasoning is truly language-independent.
Also tests whether Qwen3's multilingual training creates cross-lingual encoding.

In [ ]:
import subprocess, sys
from pathlib import Path

WORKSPACE = Path("/workspace/13-4-2026")
REPO_DIR = WORKSPACE / "legibility"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "-b", "13-4-2026",
        "https://github.com/JackHopkins/legibility.git",
        str(REPO_DIR)
    ], check=True)

sys.path.insert(0, str(REPO_DIR))
from lib.config import *

for d in [CACHE_DIR, COT_CACHE, PARAPHRASE_CACHE, PREFILL_CACHE, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
import json
from tqdm import tqdm
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from lib.prompts import TRANSLATE_TO_CHINESE_PROMPT, TRANSLATE_TO_ENGLISH_PROMPT

# Load COTs
cots = []
for p in sorted(COT_CACHE.glob("*.json")):
    cots.append(json.loads(p.read_text()))
print(f"Loaded {len(cots)} COTs")

## Phase 1: Translate English -> Chinese

In [ ]:
print(f"Loading {PARAPHRASER_MODEL} for translation...")
llm = LLM(model=PARAPHRASER_MODEL, dtype="bfloat16", max_model_len=4096)
tokenizer = AutoTokenizer.from_pretrained(PARAPHRASER_MODEL)
sampling = SamplingParams(temperature=TEMPERATURE, max_tokens=MAX_COT_TOKENS)

In [ ]:
# English -> Chinese
condition_zh = "translate_zh"
done_zh = {int(p.stem.split("_")[-1]) for p in PARAPHRASE_CACHE.glob(f"{condition_zh}_*.json")}
todo_zh = [c for c in cots if c["problem_id"] not in done_zh]
print(f"EN->ZH: {len(done_zh)} done, {len(todo_zh)} remaining")

for i in tqdm(range(0, len(todo_zh), CHUNK_SIZE), desc="EN->ZH"):
    chunk = todo_zh[i:i + CHUNK_SIZE]
    prompts = [
        tokenizer.apply_chat_template(
            [{"role": "user", "content": TRANSLATE_TO_CHINESE_PROMPT.format(cot_text=c["cot_text"])}],
            tokenize=False, add_generation_prompt=True
        ) for c in chunk
    ]
    outputs = llm.generate(prompts, sampling)
    for c, output in zip(chunk, outputs):
        result = {
            "problem_id": c["problem_id"],
            "condition": condition_zh,
            "original_cot": c["cot_text"],
            "paraphrased_cot": output.outputs[0].text.strip(),
        }
        (PARAPHRASE_CACHE / f"{condition_zh}_{c['problem_id']}.json").write_text(json.dumps(result))

## Phase 2: Translate Chinese -> English

In [ ]:
# Load Chinese translations
zh_lookup = {}
for p in PARAPHRASE_CACHE.glob(f"{condition_zh}_*.json"):
    r = json.loads(p.read_text())
    zh_lookup[r["problem_id"]] = r["paraphrased_cot"]

condition_rt = "roundtrip_zh"
done_rt = {int(p.stem.split("_")[-1]) for p in PARAPHRASE_CACHE.glob(f"{condition_rt}_*.json")}
todo_rt = [c for c in cots if c["problem_id"] not in done_rt and c["problem_id"] in zh_lookup]
print(f"ZH->EN: {len(done_rt)} done, {len(todo_rt)} remaining")

for i in tqdm(range(0, len(todo_rt), CHUNK_SIZE), desc="ZH->EN"):
    chunk = todo_rt[i:i + CHUNK_SIZE]
    prompts = [
        tokenizer.apply_chat_template(
            [{"role": "user", "content": TRANSLATE_TO_ENGLISH_PROMPT.format(cot_text=zh_lookup[c["problem_id"]])}],
            tokenize=False, add_generation_prompt=True
        ) for c in chunk
    ]
    outputs = llm.generate(prompts, sampling)
    for c, output in zip(chunk, outputs):
        result = {
            "problem_id": c["problem_id"],
            "condition": condition_rt,
            "original_cot": c["cot_text"],
            "paraphrased_cot": output.outputs[0].text.strip(),
        }
        (PARAPHRASE_CACHE / f"{condition_rt}_{c['problem_id']}.json").write_text(json.dumps(result))

del llm, tokenizer
import gc; gc.collect()
import torch; torch.cuda.empty_cache()

## Phase 3: Prefill PRIMARY_MODEL with round-tripped COTs

In [ ]:
from lib.data import load_gsm8k
from lib.prefill import run_prefill_condition

dataset = load_gsm8k()

# Load round-tripped COTs
rt_lookup = {}
for p in PARAPHRASE_CACHE.glob(f"{condition_rt}_*.json"):
    r = json.loads(p.read_text())
    rt_lookup[r["problem_id"]] = r["paraphrased_cot"]
print(f"Round-tripped COTs: {len(rt_lookup)}")

print(f"Loading {PRIMARY_MODEL}...")
llm = LLM(model=PRIMARY_MODEL, dtype="bfloat16", max_model_len=4096)

run_prefill_condition(llm, "roundtrip_zh_self", PRIMARY_MODEL, dataset, rt_lookup)

del llm
import gc; gc.collect()
import torch; torch.cuda.empty_cache()

## Analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def get_acc(condition):
    results = []
    for p in PREFILL_CACHE.glob(f"{condition}_*.json"):
        r = json.loads(p.read_text())
        results.append(r["predicted_answer"] == r["gold_answer"])
    return np.mean(results) if results else None

conditions = ["no_cot", "self_prefill", "paraphrase_self", "roundtrip_zh_self", "paraphrase_cross"]
labels = ["No COT", "Self prefill", "Paraphrase\nself", "Round-trip\n(ZH)", "Paraphrase\ncross"]
colors = ["#95a5a6", "#8e44ad", "#2980b9", "#e74c3c", "#27ae60"]

accs = [get_acc(c) for c in conditions]
for c, a in zip(conditions, accs):
    print(f"{c:25s}: {a:.3f}" if a else f"{c:25s}: N/A")

no_cot_acc = accs[0]
self_acc = accs[1]
total_value = self_acc - no_cot_acc if (self_acc and no_cot_acc) else 0
rt_acc = accs[3]
L_rt = ((rt_acc - no_cot_acc) / total_value) if (rt_acc and total_value > 0) else None
print(f"\nRound-trip legibility L = {L_rt:.3f}" if L_rt else "\nRound-trip legibility L = N/A")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

valid = [(l, a, c) for l, a, c in zip(labels, accs, colors) if a is not None]
if valid:
    vl, va, vc = zip(*valid)
    bars = ax.bar(range(len(valid)), va, color=vc, edgecolor="black", linewidth=0.5)
    for bar, val in zip(bars, va):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{val:.3f}", ha="center", fontsize=10)
    ax.set_xticks(range(len(valid)))
    ax.set_xticklabels(vl)

ax.set_ylabel("Accuracy")
ax.set_title("Bilingual Round-Trip: Accuracy Comparison", fontsize=13, fontweight="bold")
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "bilingual_roundtrip.png", dpi=200, bbox_inches="tight")
plt.show()

# Show sample round-trip
sample = json.loads(next(PARAPHRASE_CACHE.glob(f"{condition_rt}_*.json")).read_text())
print("\n--- Sample round-trip ---")
print(f"Original (first 200): {sample['original_cot'][:200]}")
print(f"Round-tripped (first 200): {sample['paraphrased_cot'][:200]}")